In [1]:
import requests

In [2]:
url = "https://api.open-meteo.com/v1/forecast"

cities = {
    "Nyeri":   (-0.4201, 36.9476),
    "Nairobi": (-1.2921, 36.8219),
    "Mombasa": (-4.0435, 39.6682),
    "Kisumu":  (-0.0917, 34.7680),
}

raw_data = {}

for city, (lat, lon) in cities.items():
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": "Africa/Nairobi",
        "forecast_days": 7,
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()   # stops with an error if the status isn't 2xx
    raw_data[city] = response.json()
    print(city, "->", response.status_code)

Nyeri -> 200
Nairobi -> 200
Mombasa -> 200
Kisumu -> 200


In [3]:
print(raw_data['Kisumu'].keys())
print(raw_data["Kisumu"]["daily"])

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
{'time': ['2026-11-22', '2026-11-23', '2026-11-24', '2026-11-25', '2026-11-26', '2026-11-27', '2026-11-28'], 'temperature_2m_max': [29.8, 29.7, 29.8, 30.0, 28.2, 30.3, 30.2], 'temperature_2m_min': [20.4, 18.6, 19.5, 19.9, 20.2, 18.2, 19.3], 'precipitation_sum': [10.8, 0.8, 0.4, 0.3, 2.1, 0.9, 0.0]}


In [4]:
import pandas as pd

frames = []

for city, payload in raw_data.items():
    city_df = pd.DataFrame(payload["daily"])
    city_df.insert(0, "city", city)
    frames.append(city_df)

df = pd.concat(frames, ignore_index=True)
print(df.shape)
df.head(10)

(28, 5)


,city,time,temperature_2m_max,temperature_2m_min,precipitation_sum
0,Nyeri,2026-11-22,20.3,15.4,6.7
1,Nyeri,2026-11-23,20.6,14.5,9.6
2,Nyeri,2026-11-24,21.5,14.4,3.4
3,Nyeri,2026-11-25,22.4,13.4,7.1
4,Nyeri,2026-11-26,23.0,11.6,4.5
5,Nyeri,2026-11-27,20.2,13.6,9.9
6,Nyeri,2026-11-28,22.7,12.3,2.4
7,Nairobi,2026-11-22,22.9,16.1,5.0
8,Nairobi,2026-11-23,20.2,16.1,13.4
9,Nairobi,2026-11-24,23.6,16.3,3.4


In [5]:
df = df.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max",
    "temperature_2m_min": "temp_min",
    "precipitation_sum": "rainfall_mm",
})

before = len(df)
df_clean = df.dropna().copy()

df_clean["temp_range"] = (df_clean["temp_max"] - df_clean["temp_min"]).round(1)

def rain_level(mm):
    if mm == 0:
        return "Dry"
    elif mm < 5:
        return "Light"
    elif mm < 20:
        return "Moderate"
    return "Heavy"

df_clean["rain_level"] = df_clean["rainfall_mm"].apply(rain_level)

print("Before cleaning:", before)
print("After cleaning:", len(df_clean))
df_clean.head(10)

Before cleaning: 28
After cleaning: 28


,city,date,temp_max,temp_min,rainfall_mm,temp_range,rain_level
0,Nyeri,2026-11-22,20.3,15.4,6.7,4.9,Moderate
1,Nyeri,2026-11-23,20.6,14.5,9.6,6.1,Moderate
2,Nyeri,2026-11-24,21.5,14.4,3.4,7.1,Light
3,Nyeri,2026-11-25,22.4,13.4,7.1,9.0,Moderate
4,Nyeri,2026-11-26,23.0,11.6,4.5,11.4,Light
5,Nyeri,2026-11-27,20.2,13.6,9.9,6.6,Moderate
6,Nyeri,2026-11-28,22.7,12.3,2.4,10.4,Light
7,Nairobi,2026-11-22,22.9,16.1,5.0,6.8,Moderate
8,Nairobi,2026-11-23,20.2,16.1,13.4,4.1,Moderate
9,Nairobi,2026-11-24,23.6,16.3,3.4,7.3,Light


In [6]:
import sqlite3

conn = sqlite3.connect("weather.db")  
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS daily_weather (
        city        TEXT,
        date        TEXT,
        temp_max    REAL,
        temp_min    REAL,
        rainfall_mm REAL,
        temp_range  REAL,
        rain_level  TEXT,
        PRIMARY KEY (city, date)
    )
""")
conn.commit()

In [7]:
rows = df_clean[
    ["city", "date", "temp_max", "temp_min", "rainfall_mm", "temp_range", "rain_level"]
].values.tolist()

cursor.executemany("""
    INSERT OR REPLACE INTO daily_weather
    (city, date, temp_max, temp_min, rainfall_mm, temp_range, rain_level)
    VALUES (?, ?, ?, ?, ?, ?, ?)
""", rows)

conn.commit()
print(f"Pipeline complete. {len(rows)} rows loaded into weather.db")

Pipeline complete. 28 rows loaded into weather.db


In [8]:
print(pd.read_sql("SELECT COUNT(*) AS total_rows FROM daily_weather", conn))

query = """
    SELECT city,
           ROUND(AVG(temp_max), 1)   AS avg_high,
           ROUND(AVG(temp_min), 1)   AS avg_low,
           ROUND(SUM(rainfall_mm), 1) AS total_rain_mm
    FROM daily_weather
    GROUP BY city
    ORDER BY total_rain_mm DESC
"""
print(pd.read_sql(query, conn))

   total_rows
0          28
      city  avg_high  avg_low  total_rain_mm
0  Nairobi      23.3     16.4           46.9
1    Nyeri      21.5     13.6           43.6
2  Mombasa      30.7     24.6           16.2
3   Kisumu      29.7     19.4           15.3


In [9]:
df_clean.to_csv('weather.csv', index=False)

print("Data saved to 'weather.csv'")
print(f"Repos loaded into weather.csv: {len(df_clean)}")

Data saved to 'weather.csv'
Repos loaded into weather.csv: 28
